# ArcGIS Online Item Terms of Use Editor

**Welcome!**  
This notebook helps you scan, review, and update ArcGIS Online items at scale. It focuses on the Terms of Use section, stored in the `licenseInfo` field, and looks for target text or HTML that you want to replace. It was originally built to find broken logo references and replace them with current content, but the same workflow can be adapted for other patterns in an item's description.

To keep the interface simple, most of the logic lives in `helper_functions.py`.
If you run the portable notebook variant, bundled assets (`helper_functions.py` and `Esri_ToU.html`) are bootstrapped into `/arcgis/home/notebook_outputs` in ArcGIS Online (or `./notebook_outputs` in local runs).
For this source notebook, make sure `helper_functions.py` is available from your runtime path (for example `/arcgis/home`, `/arcgis/home/notebook_outputs`, or the current working directory).

**Where this notebook can run**  
- ArcGIS Online Notebook (JupyterLab-style).
- VS Code on macOS with a local Jupyter kernel.
- VS Code on Windows with a local Jupyter kernel.

**How to use this notebook**  
- Start with **1. Setup and authenticate** to connect to your organization.
- Run **2. Scan your content** to find matching terms.
- Save the scan outputs, optionally run a secondary scan, and review exact matches if needed.
- Build a **dry-run plan** to see exactly what would change before anything is updated.
- Use the dry-run output to create an HTML review report for side-by-side comparison.
- Commit updates only after you have reviewed the dry-run and report outputs.

**Notes**  
- Organization-wide scans can take time, especially in large orgs, so progress messages are shown as users are processed.
- The workflow is designed to be safe by default: review first, then update.
- If you need to restart, restart the kernel and begin again at Step 1.
- In ArcGIS Online, you can use **View > Collapse All Code** for a cleaner, more user-friendly layout.

**TL;DR**

In [ ]:
# Run this cell to display Notebook details
from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does**  
- Authenticates to ArcGIS Online
- Scans an entire Organization's Item Description page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames: `scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user-friendly side-by-side HTML review report for visual QA.
- Applies updates only after explicit confirmation, then exports success and error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))

## 1. Setup and authenticate
Connect to ArcGIS Online and initialize the notebook environment.

In [ ]:
# When running in ArcGIS Online, you can select View > Collapse All Code in the toolbar above to hide the code for a cleaner view.
print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

# Support local VS Code and ArcGIS Online paths; prefer bootstrapped notebook_outputs first.
NOTEBOOK_DIR = Path.cwd()
CANDIDATE_HELPER_DIRS = [
    NOTEBOOK_DIR / "notebook_outputs",
    NOTEBOOK_DIR,
    Path("/arcgis/home/notebook_outputs"),
    Path("/arcgis/home"),
]
for p in CANDIDATE_HELPER_DIRS:
    helper_file = p / "helper_functions.py"
    if helper_file.exists() and str(p) not in sys.path:
        sys.path.append(str(p))

from helper_functions import (
    detect_environment,
    default_output_dir_str,
    default_output_path_str,
    initialize_ui,
    set_runtime_context,
    bind_button_with_status,
    setup_notebook_btn,
    run_primary_scan_btn,
    save_scan_outputs_btn,
    load_previous_scan_btn,
    run_secondary_scan_btn,
    run_strict_match_filter_btn,
    run_dry_run_with_file_btn,
    create_report_btn,
    export_dry_run_btn,
    apply_updates_btn,
    export_final_results_btn,
    OFFICIAL_TOU_HTML_FILE,
 )

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {
    "gis": None,
    "matches_df": None,
    "errors_df": None,
    "all_items_df": None,
    "TARGET_STRINGS": [],
    "output_dir": default_output_dir_str(),
    "official_tou_html_file": OFFICIAL_TOU_HTML_FILE,
}
set_runtime_context(context)

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")
print(f"Default output folder: {context['output_dir']}")

output1 = initialize_ui("output")
context["output1"] = output1

# Create widgets
btn1 = initialize_ui("button", description="Setup Notebook", width="200px")
status1 = widgets.HTML(value="")
context["status1"] = status1
display(widgets.HBox([btn1, status1]))
bind_button_with_status(
    btn1,
    setup_notebook_btn,
    "status1",
    "Setup in progress... please wait.",
    "Setup complete.",
    "Setup failed. See details below.",
    output_key="output1",
)
display(output1)

Initializing...
Currently running in VSCode Notebook environment
Default output folder: /Users/davi6569/Documents/GitHub/AGO-item-description-editor/notebook_outputs


Button(description='Setup Notebook', layout=Layout(height='40px', width='200px'), style=ButtonStyle())

Output()

## 2. Scan your content
Search your content for the text strings and/or HTML entered below.

In [ ]:
# Cell 2: Define terms and scan org content for licenseInfo matches
output2 = initialize_ui("output")
context["output2"] = output2

help2 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "<strong>Enter text or HTML to find in the Terms of Use section.</strong> "
        "Use CSV-style input (comma-separated).<br>"
        "Wrap text with spaces in double quotes, for example: "
        "&quot;&lt;a href=https://example.com&gt;link&lt;/a&gt;&quot;.<br>"
        "Formatting will automatically be bundled for proceesing when you click the button."
        "</div>"
    )
)

input2 = initialize_ui(
    "textarea",
    value='https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png, esrilogo',
    description="",
    width="700px",
    height="70px",
)
context["input2"] = input2
btn2 = initialize_ui("button", description="Scan for items", width="200px")
status2 = widgets.HTML(value="")
context["status2"] = status2

display(widgets.VBox([help2, input2, widgets.HBox([btn2, status2]), output2]))

bind_button_with_status(
    btn2,
    run_primary_scan_btn,
    "status2",
    "Scan in progress... please wait.",
    "Scan complete.",
    "Scan failed. See details below.",
    output_key="output2",
)

## 3. Save scan results
Save the latest scan output to CSV files. You can rename the files or change where they are written.

In [ ]:
# Cell 3: Save latest scan outputs to CSV
output3 = initialize_ui("output")
context["output3"] = output3
input3_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="700px",
)
context["input3_matches"] = input3_matches
input3_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["input3_errors"] = input3_errors
input3_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="700px",
)
context["input3_all_items"] = input3_all_items
btn3 = initialize_ui("button", description="Save files")
status3 = widgets.HTML(value="")
context["status3"] = status3
display(widgets.VBox([input3_matches, input3_errors, input3_all_items, widgets.HBox([btn3, status3]), output3]))

bind_button_with_status(
    btn3,
    save_scan_outputs_btn,
    "status3",
    "Save in progress... please wait.",
    "Save complete.",
    "Save failed. See details below.",
    output_key="output3",
)

## 4. Optionally reload results from a previous scan
Load previously saved CSV files so you can continue working without running a new scan.

In [ ]:
# Cell 4: Optionally load results from a previous scan
output4 = initialize_ui("output")
context["output4"] = output4

input4_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="900px",
)
context["input4_matches"] = input4_matches
input4_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="900px",
)
context["input4_errors"] = input4_errors
input4_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="900px",
)
context["input4_all_items"] = input4_all_items
btn4 = initialize_ui("button", description="Load previous scan files", width="240px")
status4 = widgets.HTML(value="")
context["status4"] = status4

display(
    widgets.VBox([
        input4_matches,
        input4_errors,
        input4_all_items,
        widgets.HBox([btn4, status4]),
        output4,
    ])
)

bind_button_with_status(
    btn4,
    load_previous_scan_btn,
    "status4",
    "Load in progress... please wait.",
    "Load complete.",
    "Load failed. See details below.",
    output_key="output4",
)

## 5. Secondary scan
This cell saves you time if you want to search additional terms.
If you want to search again, click the SECONDARY_SCAN check box and add the new terms to NEW_TERMS and run the cell below

In [ ]:
# Cell 5: Optional secondary scan using new terms and excluding previous matches
output5 = initialize_ui("output")
context["output5"] = output5
checkbox5 = initialize_ui("checkbox", description="Run secondary scan with new search terms?", value=False)
context["checkbox5"] = checkbox5
help5 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "As above, use CSV-style input separated by commas.<br>"
        "Wrap text with spaces in double quotes, for example: &quot;text with spaces&quot;."
        "</div>"
    )
)
input5 = initialize_ui(
    "textarea",
    value='https://www.esri.com/content/dam/esrisites/en-us/common/logos/esri-logo.jpg',
    description="",
    width="700px",
    height="70px",
)
context["input5"] = input5

btn5 = initialize_ui("button", description="Run secondary scan")
status5 = widgets.HTML(value="")
context["status5"] = status5
display(widgets.VBox([checkbox5, help5, input5, widgets.HBox([btn5, status5]), output5]))

bind_button_with_status(
    btn5,
    run_secondary_scan_btn,
    "status5",
    "Secondary scan in progress... please wait.",
    "Secondary scan complete.",
    "Secondary scan failed. See details below.",
    output_key="output5",
)

## 6. Optionally narrow your original query

In [ ]:
# Cell 6: Optionally filter the scan result to rows containing the exact text below
output6 = initialize_ui("output")
context["output6"] = output6
input6 = initialize_ui(
    "text",
    value="https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png",
    description="Exact text:",
    width="700px",
)
context["input6"] = input6
btn6 = initialize_ui("button", description="Filter exact matches", width="200px")
status6 = widgets.HTML(value="")
context["status6"] = status6
display(widgets.VBox([input6, widgets.HBox([btn6, status6]), output6]))

bind_button_with_status(
    btn6,
    run_strict_match_filter_btn,
    "status6",
    "Filter in progress... please wait.",
    "Filter complete.",
    "Filter failed. See details below.",
    output_key="output6",
)

## 7. Dry run

In [ ]:
# Cell 7: Do a dry-run before making any changes. Modify the input file to use your own custom HTML.
output7 = initialize_ui("output")
context["output7"] = output7
input7 = initialize_ui(
    "text",
    value=context.get("official_tou_html_file", OFFICIAL_TOU_HTML_FILE),
    description="Input HTML file:",
    width="900px",
)
context["input7"] = input7
btn7 = initialize_ui("button", description="Build dry run", width="180px")
status7 = widgets.HTML(value="")
context["status7"] = status7

display(widgets.VBox([input7, widgets.HBox([btn7, status7]), output7]))

bind_button_with_status(
    btn7,
    run_dry_run_with_file_btn,
    "status7",
    "Dry run in progress... please wait.",
    "Dry run complete.",
    "Dry run failed. See details below.",
    output_key="output7",
)

## 8. Export dry run results

In [ ]:
# Cell 8: Export the dry-run plan for record-keeping and review
output8 = initialize_ui("output")
context["output8"] = output8
input8_csv_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_results.csv"),
    description="Output CSV:",
    width="700px",
)
context["input8_csv_name"] = input8_csv_name
btn8 = initialize_ui("button", description="Export dry-run CSV", width="200px")
status8 = widgets.HTML(value="")
context["status8"] = status8
display(widgets.VBox([input8_csv_name, widgets.HBox([btn8, status8]), output8]))

bind_button_with_status(
    btn8,
    export_dry_run_btn,
    "status8",
    "Dry-run export in progress... please wait.",
    "Dry-run export complete.",
    "Dry-run export failed. See details below.",
    output_key="output8",
)

## 9. Create report

In [ ]:
# Cell 9: Generate an HTML report for review before updating items
output9 = initialize_ui("output")
context["output9"] = output9
input9_report_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_report.html"),
    description="Report HTML:",
    width="700px",
)
context["input9_report_name"] = input9_report_name
input9_selection_json = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="Filename generated when downloading IDs after review:",
    width="700px",
)
context["input9_selection_json"] = input9_selection_json
btn9 = initialize_ui("button", description="Create report", width="200px")
status9 = widgets.HTML(value="")
context["status9"] = status9

display(
    widgets.VBox([
        input9_report_name,
        input9_selection_json,
        widgets.HBox([btn9, status9]),
        output9,
    ])
)

bind_button_with_status(
    btn9,
    create_report_btn,
    "status9",
    "Report generation in progress... please wait.",
    "Report generation complete.",
    "Report generation failed. See details below.",
    output_key="output9",
)

## 10. Commit updates

In [ ]:
# Cell 10: Apply updates only after reviewing the dry run report 
output10 = initialize_ui("output")
context["output10"] = output10
input10_ids = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="JSON file with item IDs to update:",
    width="700px",
)
context["input10_ids"] = input10_ids

input10_confirm = initialize_ui(
    "text",
    value="",
    description="Type APPLY UPDATES to confirm:",
    width="700px",
)
context["input10_confirm"] = input10_confirm

btn10 = initialize_ui("button", description="Execute update", width="180px")
status10 = widgets.HTML(value="")
context["status10"] = status10
display(widgets.VBox([input10_ids, input10_confirm, widgets.HBox([btn10, status10]), output10]))

bind_button_with_status(
    btn10,
    apply_updates_btn,
    "status10",
    "Update in progress... please wait.",
    "Update complete.",
    "Update failed. See details below.",
    output_key="output10",
)

## 11. Export final results

In [ ]:
# Cell 11: Export final update results to CSV files for record-keeping
output11 = initialize_ui("output")
context["output11"] = output11
input11_success_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_successes.csv"),
    description="Success CSV:",
    width="700px",
)
context["input11_success_csv"] = input11_success_csv
input11_errors_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["input11_errors_csv"] = input11_errors_csv
btn11 = initialize_ui("button", description="Export final CSVs", width="180px")
status11 = widgets.HTML(value="")
context["status11"] = status11
display(widgets.VBox([input11_success_csv, input11_errors_csv, widgets.HBox([btn11, status11]), output11]))

bind_button_with_status(
    btn11,
    export_final_results_btn,
    "status11",
    "Final export in progress... please wait.",
    "Final export complete.",
    "Final export failed. See details below.",
    output_key="output11",
)